In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz

def show_bubble_chart(apps_csv, reviews_csv):
    # 1. Check Current Time in IST
    ist_tz = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist_tz)
    
    # 2. Time-Gate Logic: Only execute between 17:00 (5 PM) and 19:00 (7 PM)
    if not (17 <= current_time_ist.hour < 19):
        print(f"Graph is currently hidden. It is only available between 5 PM and 7 PM IST. (Current time: {current_time_ist.strftime('%I:%M %p')} IST)")
        return  # Exit the function, preventing the graph from rendering in the dashboard

    # 3. Load Data
    df_apps = pd.read_csv(apps_csv)
    df_reviews = pd.read_csv(reviews_csv)

    # 4. Clean Data
    # Clean Installs
    df_apps['Installs'] = df_apps['Installs'].astype(str).str.replace(r'[+,]', '', regex=True)
    df_apps['Installs'] = pd.to_numeric(df_apps['Installs'], errors='coerce')

    # Clean Size (MB)
    def clean_size_mb(size):
        if isinstance(size, str):
            if 'M' in size:
                return float(size.replace('M', ''))
            elif 'k' in size:
                return float(size.replace('k', '')) / 1024
        return np.nan
    df_apps['Size_MB'] = df_apps['Size'].apply(clean_size_mb)

    # Clean Reviews
    df_apps['Reviews'] = pd.to_numeric(df_apps['Reviews'], errors='coerce')
    df_apps['Rating'] = pd.to_numeric(df_apps['Rating'], errors='coerce')

    # Calculate average Sentiment Subjectivity per app
    df_reviews['Sentiment_Subjectivity'] = pd.to_numeric(df_reviews['Sentiment_Subjectivity'], errors='coerce')
    app_subjectivity = df_reviews.groupby('App')['Sentiment_Subjectivity'].mean().reset_index()

    # Merge datasets
    df_merged = pd.merge(df_apps, app_subjectivity, on='App', how='inner')
    
    # Drop duplicates in case of multiple identical app entries in the store data
    df_merged = df_merged.drop_duplicates(subset=['App'])

    # 5. Apply Complex Filters
    # Required categories (Must match Google Play formatting)
    allowed_categories = ['GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION', 'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENTS']
    
    df_filtered = df_merged[
        (df_merged['Rating'] > 3.5) &
        (df_merged['Category'].isin(allowed_categories)) &
        (df_merged['Reviews'] > 500) &
        (~df_merged['App'].str.lower().str.contains('s', na=False)) & # Name must not contain 'S' or 's'
        (df_merged['Sentiment_Subjectivity'] > 0.5) &
        (df_merged['Installs'] > 50000)
    ].copy()
    
    # Drop rows where Size_MB is NaN for plotting purposes
    df_filtered = df_filtered.dropna(subset=['Size_MB', 'Rating'])

    # 6. Apply Category Translations
    translations = {
        'BEAUTY': 'सौंदर्य',          # Hindi
        'BUSINESS': 'வணிகம்',         # Tamil
        'DATING': 'Partnersuche'      # German
    }
    df_filtered['Category'] = df_filtered['Category'].replace(translations)

    # 7. Define Colors for Plotly (Highlight Game category in Pink)
    # Get all unique categories currently in the filtered dataset
    unique_cats = df_filtered['Category'].unique()
    
    color_map = {}
    for cat in unique_cats:
        if cat == 'GAME':
            color_map[cat] = 'deeppink' # Highlight Pink
        else:
            color_map[cat] = 'lightslategray' # Default color for everything else

    # 8. Render the Interactive Bubble Chart
    fig = px.scatter(
        df_filtered,
        x='Size_MB',
        y='Rating',
        size='Installs',
        color='Category',
        color_discrete_map=color_map,
        hover_name='App',
        hover_data={'Category': True, 'Installs': True, 'Sentiment_Subjectivity': ':.2f'},
        title='App Size vs. Average Rating (Bubble Size = Installs)',
        labels={'Size_MB': 'App Size (MB)', 'Rating': 'Average Rating'},
        size_max=50 # Controls the maximum bubble size for readability
    )
    
    fig.update_layout(template='plotly_white')
    
    # Display the chart (Use st.plotly_chart(fig) if using Streamlit)
    fig.show()

# Usage
show_bubble_chart('googleplaystore.csv', 'User Reviews.csv')

Graph is currently hidden. It is only available between 5 PM and 7 PM IST. (Current time: 02:25 PM IST)
